# UAV Cross-Domain Diagnosis

This notebook is the exploratory diagnosis view for the `Crack500 -> UAV_Crack_Segmentation_Kaggle` transfer setting.

It intentionally runs on `crossdomain_all` rather than the official hold-out split, because its purpose is failure analysis and pattern discovery rather than final benchmark reporting.

It separates two reporting layers:

1. `Raw model performance`: this remains the official cross-domain baseline because it measures the model's own generalization under domain shift.
2. `Deployment-oriented postprocessed variant`: this is reported separately because it improves cluttered UAV scenes through an explicit `precision-recall` tradeoff rather than through a universal model improvement.

The notebook does five things:
1. loads the full UAV diagnosis set
2. rebuilds the frozen `U-Net` and `SegFormer-B2` checkpoints
3. reports raw baseline metrics and the optional deployment-oriented variant side by side
4. keeps a raw threshold sweep as a calibration diagnostic, not as a baseline replacement
5. visualizes representative false-positive suppression cases in cluttered UAV imagery


In [ ]:
import os
import sys

sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from dataset import CrackDataset
from loss import build_loss
from metrics import confusion_stats
from model import get_model
from postprocess import build_postprocess_config, logits_to_binary_mask

ROOT = "../UAV_Crack_Segmentation_Kaggle"
DIAGNOSIS_SPLIT = "crossdomain_all"
IMG_SIZE = 360
RAW_THRESHOLD = 0.5
DEPLOYMENT_THRESHOLD = 0.9
DEPLOYMENT_POSTPROCESS_KWARGS = {
    "min_area": 20,
    "max_fill_ratio": 0.85,
    "min_aspect_ratio": 1.0,
    "max_components": 0,
}
DEPLOYMENT_POSTPROCESS_CONFIG = build_postprocess_config(**DEPLOYMENT_POSTPROCESS_KWARGS)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = build_loss("bce_dice")
plt.rcParams["figure.dpi"] = 120

MODEL_CONFIGS = {
    "U-Net": {
        "model_name": "Unet",
        "encoder_name": "resnet34",
        "encoder_weights": "imagenet",
        "checkpoint_path": os.path.join("..", "checkpoints", "unet_ablate_aug_mild_360_fgcrop.pth"),
        "batch_size": 16,
    },
    "SegFormer-B2": {
        "model_name": "segformer-b2",
        "encoder_name": "resnet34",
        "encoder_weights": "imagenet",
        "checkpoint_path": os.path.join("..", "checkpoints", "segformer_b2_plain_360.pth"),
        "batch_size": 8,
    },
}

print(f"Using device: {device}")
print(f"Dataset root: {ROOT}")
print(f"Diagnosis split: {DIAGNOSIS_SPLIT}")
print(f"Raw baseline threshold: {RAW_THRESHOLD}")
print(
    "Deployment variant: "
    f"threshold={DEPLOYMENT_THRESHOLD}, "
    f"postprocess={DEPLOYMENT_POSTPROCESS_KWARGS}"
)
for label, cfg in MODEL_CONFIGS.items():
    print(f"{label:13s} -> {cfg['checkpoint_path']}")

In [ ]:
test_dataset = CrackDataset(ROOT, split=DIAGNOSIS_SPLIT, img_size=IMG_SIZE)

raw_image, raw_mask = test_dataset.get_raw(0)
print(f"Loaded UAV diagnosis dataset with {len(test_dataset)} samples.")
print(f"Sample 0 raw image shape: {raw_image.shape}")
print(f"Sample 0 raw mask shape: {raw_mask.shape}")
print(f"Sample 0 foreground ratio: {raw_mask.mean():.4f}")

In [ ]:
def load_model_from_config(cfg):
    checkpoint_path = cfg["checkpoint_path"]
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

    model = get_model(
        model_name=cfg["model_name"],
        encoder_name=cfg["encoder_name"],
        encoder_weights=cfg["encoder_weights"],
        in_channels=3,
        classes=1,
    )
    state_dict = torch.load(checkpoint_path, map_location=device, weights_only=True)
    model.load_state_dict(state_dict)
    model = model.to(device)
    model.eval()
    return model


models = {label: load_model_from_config(cfg) for label, cfg in MODEL_CONFIGS.items()}
print("Loaded models:", ", ".join(models.keys()))

In [ ]:
def format_value(value, round_digits=4):
    if isinstance(value, float):
        return f"{value:.{round_digits}f}"
    return str(value)


def print_table(rows, columns, round_digits=4):
    if not rows:
        print("<empty>")
        return

    widths = []
    for col in columns:
        max_len = len(col)
        for row in rows:
            max_len = max(max_len, len(format_value(row[col], round_digits=round_digits)))
        widths.append(max_len)

    header = " | ".join(col.ljust(width) for col, width in zip(columns, widths))
    divider = "-+-".join("-" * width for width in widths)
    print(header)
    print(divider)
    for row in rows:
        print(
            " | ".join(
                format_value(row[col], round_digits=round_digits).ljust(width)
                for col, width in zip(columns, widths)
            )
        )


def make_overlay(image_np, mask_np, color=(255, 0, 0), alpha=0.45):
    overlay = image_np.copy().astype(np.float32)
    color_arr = np.array(color, dtype=np.float32)
    overlay[mask_np == 1] = overlay[mask_np == 1] * (1 - alpha) + color_arr * alpha
    return overlay.clip(0, 255).astype(np.uint8)


def make_error_map(pred_mask, gt_mask):
    error_rgb = np.zeros((*gt_mask.shape, 3), dtype=np.uint8)
    error_rgb[(pred_mask == 1) & (gt_mask == 1)] = [0, 200, 0]
    error_rgb[(pred_mask == 1) & (gt_mask == 0)] = [220, 0, 0]
    error_rgb[(pred_mask == 0) & (gt_mask == 1)] = [255, 215, 0]
    return error_rgb


def predict_one(model, index, threshold, postprocess_config=None):
    raw_image, raw_mask = test_dataset.get_raw(index)

    image_t, mask_t = test_dataset[index]
    image_t = image_t.unsqueeze(0).to(device)
    mask_t = mask_t.unsqueeze(0).unsqueeze(1).float().to(device)

    with torch.no_grad():
        logits = model(image_t)
        probs = torch.sigmoid(logits)
        pred_mask_t = logits_to_binary_mask(
            logits,
            threshold=threshold,
            postprocess_config=postprocess_config,
        )
        loss = criterion(logits, mask_t).item()
        metrics = confusion_stats(
            logits,
            mask_t,
            threshold=threshold,
            postprocess_config=postprocess_config,
        )

    raw_h, raw_w = raw_mask.shape
    pred_mask = F.interpolate(
        pred_mask_t,
        size=(raw_h, raw_w),
        mode="nearest",
    )[0, 0].cpu().numpy().astype(np.uint8)

    prob_map = F.interpolate(
        probs,
        size=(raw_h, raw_w),
        mode="bilinear",
        align_corners=False,
    )[0, 0].cpu().numpy()

    return {
        "image": raw_image,
        "gt_mask": raw_mask,
        "pred_mask": pred_mask,
        "prob_map": prob_map,
        "gt_overlay": make_overlay(raw_image, raw_mask),
        "pred_overlay": make_overlay(raw_image, pred_mask),
        "error_map": make_error_map(pred_mask, raw_mask),
        "loss": loss,
        **metrics,
        "gt_fg_ratio": float(raw_mask.mean()),
        "pred_fg_ratio": float(pred_mask.mean()),
        "prob_mean": float(prob_map.mean()),
    }


def evaluate_model(model, batch_size, threshold, postprocess_config=None):
    loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    total_loss = 0.0
    total_iou = 0.0
    total_f1 = 0.0
    total_precision = 0.0
    total_recall = 0.0

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.unsqueeze(1).float().to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)
            batch_metrics = confusion_stats(
                outputs,
                masks,
                threshold=threshold,
                postprocess_config=postprocess_config,
            )

            batch_size_now = images.size(0)
            total_loss += loss.item() * batch_size_now
            total_iou += batch_metrics["iou"] * batch_size_now
            total_f1 += batch_metrics["f1"] * batch_size_now
            total_precision += batch_metrics["precision"] * batch_size_now
            total_recall += batch_metrics["recall"] * batch_size_now

    n = len(test_dataset)
    return {
        "loss": total_loss / n,
        "iou": total_iou / n,
        "f1": total_f1 / n,
        "precision": total_precision / n,
        "recall": total_recall / n,
    }

## Summary Table

The table below should be read in two layers:
- `Raw baseline` is the official cross-domain result.
- `Deployment-oriented variant` is a separate target-domain operating point designed to reduce false positives in cluttered UAV scenes.

In [ ]:
summary_rows = []
variant_configs = [
    {
        "report_group": "Raw model performance",
        "model": None,
        "variant": "Raw baseline",
        "threshold": RAW_THRESHOLD,
        "postprocess": "none",
        "postprocess_config": None,
    },
    {
        "report_group": "Deployment-oriented postprocessed variant",
        "model": None,
        "variant": "Deployment-oriented variant",
        "threshold": DEPLOYMENT_THRESHOLD,
        "postprocess": "connected-components",
        "postprocess_config": DEPLOYMENT_POSTPROCESS_CONFIG,
    },
]

for label, model in models.items():
    for variant_cfg in variant_configs:
        stats = evaluate_model(
            model=model,
            batch_size=MODEL_CONFIGS[label]["batch_size"],
            threshold=variant_cfg["threshold"],
            postprocess_config=variant_cfg["postprocess_config"],
        )
        summary_rows.append({
            "report_group": variant_cfg["report_group"],
            "model": label,
            "variant": variant_cfg["variant"],
            "threshold": variant_cfg["threshold"],
            "postprocess": variant_cfg["postprocess"],
            **stats,
        })

print_table(
    summary_rows,
    columns=["report_group", "model", "variant", "threshold", "postprocess", "loss", "iou", "f1", "precision", "recall"],
)

## Raw Threshold Sweep

This sweep is intentionally performed on raw predictions only. Its purpose is to test calibration sensitivity, not to redefine the baseline.

In [ ]:
thresholds = np.arange(0.1, 1.0, 0.1)
sweep_rows = []

for label, model in models.items():
    for thr in thresholds:
        stats = evaluate_model(
            model=model,
            batch_size=MODEL_CONFIGS[label]["batch_size"],
            threshold=float(thr),
            postprocess_config=None,
        )
        sweep_rows.append({
            "model": label,
            "threshold": float(thr),
            **stats,
        })

best_iou_rows = []
for label in MODEL_CONFIGS:
    model_rows = [row for row in sweep_rows if row["model"] == label]
    best_iou_rows.append(max(model_rows, key=lambda row: row["iou"]))

print("Best raw-threshold operating point by IoU")
print_table(best_iou_rows, columns=["model", "threshold", "iou", "f1", "precision", "recall"])

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
metric_names = ["iou", "f1", "precision", "recall"]
titles = ["Raw Threshold Sweep: IoU", "Raw Threshold Sweep: F1", "Raw Threshold Sweep: Precision", "Raw Threshold Sweep: Recall"]
for ax, metric_name, title in zip(axes.flat, metric_names, titles):
    for label in MODEL_CONFIGS:
        model_rows = [row for row in sweep_rows if row["model"] == label]
        ax.plot(
            [row["threshold"] for row in model_rows],
            [row[metric_name] for row in model_rows],
            marker="o",
            label=label,
        )
    ax.set_title(title)
    ax.set_xlabel("Threshold")
    ax.set_ylabel(metric_name.capitalize())
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()

## Candidate False-Positive Suppression Cases

To keep the qualitative review focused, we rank UAV samples where `SegFormer-B2` loses a large amount of predicted foreground after postprocessing. These are good candidates for non-crack clutter or object-induced false positives.

In [ ]:
segformer_case_rows = []
for idx in range(len(test_dataset)):
    raw_sample = predict_one(
        models["SegFormer-B2"],
        idx,
        threshold=RAW_THRESHOLD,
        postprocess_config=None,
    )
    deploy_sample = predict_one(
        models["SegFormer-B2"],
        idx,
        threshold=DEPLOYMENT_THRESHOLD,
        postprocess_config=DEPLOYMENT_POSTPROCESS_CONFIG,
    )
    segformer_case_rows.append({
        "index": idx,
        "gt_fg_ratio": raw_sample["gt_fg_ratio"],
        "raw_iou": raw_sample["iou"],
        "deploy_iou": deploy_sample["iou"],
        "raw_precision": raw_sample["precision"],
        "deploy_precision": deploy_sample["precision"],
        "raw_recall": raw_sample["recall"],
        "deploy_recall": deploy_sample["recall"],
        "raw_pred_fg_ratio": raw_sample["pred_fg_ratio"],
        "deploy_pred_fg_ratio": deploy_sample["pred_fg_ratio"],
        "pred_fg_drop": raw_sample["pred_fg_ratio"] - deploy_sample["pred_fg_ratio"],
        "iou_delta": deploy_sample["iou"] - raw_sample["iou"],
        "precision_delta": deploy_sample["precision"] - raw_sample["precision"],
        "recall_delta": deploy_sample["recall"] - raw_sample["recall"],
    })

fp_case_candidates = sorted(
    segformer_case_rows,
    key=lambda row: (row["pred_fg_drop"], row["precision_delta"], row["raw_pred_fg_ratio"]),
    reverse=True,
)[:10]

print_table(
    fp_case_candidates,
    columns=[
        "index",
        "gt_fg_ratio",
        "raw_pred_fg_ratio",
        "deploy_pred_fg_ratio",
        "pred_fg_drop",
        "raw_iou",
        "deploy_iou",
        "raw_precision",
        "deploy_precision",
        "raw_recall",
        "deploy_recall",
    ],
)

fp_case_indices = [row["index"] for row in fp_case_candidates[:5]]
print("Selected qualitative indices:", fp_case_indices)

## Qualitative Review: SegFormer-B2

`Green = true positive`, `red = false positive`, `yellow = false negative`.

The left half shows the raw baseline. The right half shows the deployment-oriented postprocessed variant on the same UAV samples.

In [ ]:
num_rows = len(fp_case_indices)
fig, axes = plt.subplots(num_rows, 6, figsize=(18, 3.4 * num_rows))

if num_rows == 1:
    axes = np.expand_dims(axes, axis=0)

column_titles = [
    "Image",
    "GT Overlay",
    "Raw Overlay",
    "Raw Error",
    "Deployment Overlay",
    "Deployment Error",
]
for col, title in enumerate(column_titles):
    axes[0, col].set_title(title, fontsize=11)

for row, idx in enumerate(fp_case_indices):
    raw_sample = predict_one(
        models["SegFormer-B2"],
        idx,
        threshold=RAW_THRESHOLD,
        postprocess_config=None,
    )
    deploy_sample = predict_one(
        models["SegFormer-B2"],
        idx,
        threshold=DEPLOYMENT_THRESHOLD,
        postprocess_config=DEPLOYMENT_POSTPROCESS_CONFIG,
    )

    axes[row, 0].imshow(raw_sample["image"])
    axes[row, 1].imshow(raw_sample["gt_overlay"])
    axes[row, 2].imshow(raw_sample["pred_overlay"])
    axes[row, 3].imshow(raw_sample["error_map"])
    axes[row, 4].imshow(deploy_sample["pred_overlay"])
    axes[row, 5].imshow(deploy_sample["error_map"])

    axes[row, 0].set_ylabel(
        f"idx={idx}\n"
        f"GT fg={raw_sample['gt_fg_ratio']:.3f}\n"
        f"Raw fg={raw_sample['pred_fg_ratio']:.3f}\n"
        f"Dep fg={deploy_sample['pred_fg_ratio']:.3f}",
        rotation=0,
        labelpad=45,
        va="center",
        fontsize=9,
    )
    axes[row, 2].set_xlabel(
        f"raw IoU={raw_sample['iou']:.3f}\n"
        f"P={raw_sample['precision']:.3f} R={raw_sample['recall']:.3f}",
        fontsize=9,
    )
    axes[row, 4].set_xlabel(
        f"dep IoU={deploy_sample['iou']:.3f}\n"
        f"P={deploy_sample['precision']:.3f} R={deploy_sample['recall']:.3f}",
        fontsize=9,
    )

    for ax in axes[row]:
        ax.axis("off")

plt.tight_layout()
plt.show()

## Qualitative Review: U-Net On The Same Cases

This second view checks whether the same clutter-heavy samples also confuse the weaker CNN baseline, or whether the issue is more model-specific.

In [ ]:
num_rows = len(fp_case_indices)
fig, axes = plt.subplots(num_rows, 6, figsize=(18, 3.4 * num_rows))

if num_rows == 1:
    axes = np.expand_dims(axes, axis=0)

column_titles = [
    "Image",
    "GT Overlay",
    "Raw Overlay",
    "Raw Error",
    "Deployment Overlay",
    "Deployment Error",
]
for col, title in enumerate(column_titles):
    axes[0, col].set_title(title, fontsize=11)

for row, idx in enumerate(fp_case_indices):
    raw_sample = predict_one(
        models["U-Net"],
        idx,
        threshold=RAW_THRESHOLD,
        postprocess_config=None,
    )
    deploy_sample = predict_one(
        models["U-Net"],
        idx,
        threshold=DEPLOYMENT_THRESHOLD,
        postprocess_config=DEPLOYMENT_POSTPROCESS_CONFIG,
    )

    axes[row, 0].imshow(raw_sample["image"])
    axes[row, 1].imshow(raw_sample["gt_overlay"])
    axes[row, 2].imshow(raw_sample["pred_overlay"])
    axes[row, 3].imshow(raw_sample["error_map"])
    axes[row, 4].imshow(deploy_sample["pred_overlay"])
    axes[row, 5].imshow(deploy_sample["error_map"])

    axes[row, 0].set_ylabel(
        f"idx={idx}\n"
        f"GT fg={raw_sample['gt_fg_ratio']:.3f}\n"
        f"Raw fg={raw_sample['pred_fg_ratio']:.3f}\n"
        f"Dep fg={deploy_sample['pred_fg_ratio']:.3f}",
        rotation=0,
        labelpad=45,
        va="center",
        fontsize=9,
    )
    axes[row, 2].set_xlabel(
        f"raw IoU={raw_sample['iou']:.3f}\n"
        f"P={raw_sample['precision']:.3f} R={raw_sample['recall']:.3f}",
        fontsize=9,
    )
    axes[row, 4].set_xlabel(
        f"dep IoU={deploy_sample['iou']:.3f}\n"
        f"P={deploy_sample['precision']:.3f} R={deploy_sample['recall']:.3f}",
        fontsize=9,
    )

    for ax in axes[row]:
        ax.axis("off")

plt.tight_layout()
plt.show()

## B1 Calibration Follow-Up

This notebook remains an exploratory `crossdomain_all` diagnosis notebook. Formal hold-out reporting is tracked separately on the fixed `UAV test` split.

Official `B1` hold-out reference points for `SegFormer-B2`:

- `B1 raw @ 0.5`: `IoU 0.3222`, `F1 0.4677`, `precision 0.4660`, `recall 0.5662`
- `B1 raw @ 0.7`: `IoU 0.3325`, `F1 0.4727`, `precision 0.5374`, `recall 0.5133`
- `B1 raw @ 0.9`: `IoU 0.3193`, `F1 0.4502`, `precision 0.6497`, `recall 0.4112`
- `B1 deploy`: `IoU 0.3166`, `F1 0.4469`, `precision 0.6490`, `recall 0.4071`

Takeaway:

- `B1 raw @ 0.7` is the best main-report operating point on the fixed hold-out split.
- `B1 raw @ 0.9` reaches the same high-precision regime as the old deployment switch without needing connected-component postprocessing.
- After `B1`, the postprocess switch is best kept as a comparison-only row rather than as the preferred deployment recipe.